# 07 · Feature-policy ablation: ports and the `ip` name pattern

The leakage-aware feature policy removes source and destination ports, and its substring pattern `"ip"` also removes three traffic features on NF-ToN-IoT-v2
(`MIN_IP_PKT_LEN`, `MAX_IP_PKT_LEN`, `ICMP_IPV4_TYPE`). This notebook trains the same decision trees on different feature sets and measures what each group contributes.

| Variant | Features |
|---|---|
| `A_current_policy` | the leakage-safe feature set (reference) |
| `B_plus_ip_pattern_false_positives` | A + the three features removed by the `ip` pattern |
| `C1_plus_dst_port` / `C2_plus_src_port` / `C3_plus_both_ports` | A + destination port / source port / both |
| `D_old_policy_all` | A + B's extras + both ports |
| `E_ports_only` | only the ports: if ports alone classify well, the dataset can be separated without looking at traffic behaviour |

Two models per variant, trained on the same rows: a **default decision tree** (all features, no selector) and a tree with the **saved hyper-parameters**
(with `SelectKBest`, K scaled to keep the same fraction of features). `SelectKBest` uses a linear F-test and may discard a port before the tree sees it;
the column `ports_kept` shows whether that happened, and the default-tree and ports-only rows are the main evidence.
No search is run. The train / validation / test partition is identical to the main pipeline (re-created from the labels only), and only the needed rows are read.

**Optional environment variables**: `FP_PARAMS_FILE` (a `metadata_*.json` whose `best_params` / `best_k` replace the saved ones; results go to `<dataset>_params_<name>`),
`FP_VARIANTS` (e.g. `A,B,D`; A is always included), `FP_MODELS` (`saved_params_DT`, `default_DT`), `FP_TRAIN_LIMIT`, `FP_REPEATS`.

In [ ]:
# ---- Setup: find the package root, work inside results/, import the shared code ----
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "framework.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
os.environ["PAPER_PACKAGE_ROOT"] = str(ROOT)

# >>> WHICH DATASET?  nf_ton_iot_v2 | cic_ton_iot | nf_unsw_nb15_v3
# run_all.py sets ACTIVE_DATASET; when running by hand, edit the default below
os.environ.setdefault("ACTIVE_DATASET", "nf_unsw_nb15_v3")

sys.path.insert(0, str(ROOT / "src"))
(ROOT / "results").mkdir(exist_ok=True)
os.chdir(ROOT / "results")            # every output (csv / png / pkl / headers) lands under results/

from run_settings import *            # PAPER_MODE: full | quick | smoke

In [ ]:
from eval_helpers import *
from framework import _json_default
import feature_policy as fp
import itertools, datetime, shutil, time
print("Dataset:", ACTIVE_DATASET, "| train rows per model:", f"{FP_TRAIN_LIMIT:,}", "| repeats:", FP_REPEATS)
assert Path(DEFAULT_DATASET_PATH).exists(), f"CSV not found: {DEFAULT_DATASET_PATH}"

## 1 · Column groups

In [ ]:
header = pd.read_csv(DEFAULT_DATASET_PATH, nrows=0).columns.tolist()
G = fp.classify_columns(header, TARGET_COL)
num = lambda cols: fp.numeric_columns(DEFAULT_DATASET_PATH, cols)
core = num(G["core"]); ip_fp = num(G["ip_pattern_only"])
ports_src, ports_dst, ports_oth = num(G["ports_src"]), num(G["ports_dst"]), num(G["ports_other"])
ports = ports_src + ports_dst + ports_oth

groups = pd.DataFrame([
    ("target", G["target"], "label, never a feature"), ("label_leak", G["label_leak"], "binary label -> would leak the target; always dropped"),
    ("identifiers", G["identifiers"], "IP addresses / flow ids / MACs; always dropped"), ("time", G["time"], "absolute timestamps; always dropped"),
    ("ports (src)", ports_src, "tested in C2/C3/D"), ("ports (dst)", ports_dst, "tested in C1/C3/D"), ("ports (other)", ports_oth, "tested in C3/D"),
    ("'ip'-pattern only", ip_fp, "dropped ONLY because the substring 'ip' matched -> tested in B/D"),
    ("core features", core, "leakage-safe feature set (A)")], columns=["group", "columns", "note"])
groups["n"] = groups["columns"].apply(len)
pd.set_option("display.max_colwidth", 90); display(groups[["group", "n", "columns", "note"]])

# consistency check with the model trained by notebook 01
try:
    artifact, metadata = load_artifact_and_metadata()
    same = sorted(artifact["numeric_cols"]) == sorted(core)
    print(f"\ncore features == the {len(artifact['numeric_cols'])} features of the saved model:", "yes" if same else "WARNING: NO (differences: " +
          str(sorted(set(artifact['numeric_cols']) ^ set(core))[:10]) + ")")
    BEST_K, BEST_PARAMS = int(metadata["best_k"]), dict(metadata["best_params"])
except FileNotFoundError:
    print("\n(no saved model found -> using K=30 and default parameters; run notebook 01 for the real values)")
    BEST_K, BEST_PARAMS = 30, {}
K_BASE_N = len(core)                       # the number of features that BEST_K refers to
FP_TAG = ""
_pf = os.environ.get("FP_PARAMS_FILE")
if _pf and not Path(_pf).is_absolute(): _pf = str(ROOT / _pf)     # relative paths are relative to the package folder
if _pf:
    _m = json.load(open(_pf, encoding="utf-8"))
    BEST_PARAMS, BEST_K = dict(_m["best_params"]), int(_m["best_k"])
    K_BASE_N = len(_m.get("numeric_cols", core))
    FP_TAG = "_params_" + Path(_pf).stem.replace("metadata_", "")
    print(f"\n>>> USING HYPER-PARAMETERS FROM {_pf}\n    K = {BEST_K} of {K_BASE_N} features")
print("K =", BEST_K, "(refers to", K_BASE_N, "features) | params =", BEST_PARAMS)

(Path("feature_policy_ablation") / (ACTIVE_DATASET + FP_TAG)).mkdir(parents=True, exist_ok=True)
json.dump(G, open(Path("feature_policy_ablation") / (ACTIVE_DATASET + FP_TAG) / "column_groups.json", "w"), indent=2)

# ---- variants (those whose extra group is empty are skipped) ------------------------------
VARIANTS = {"A_current_policy": core}
if ip_fp: VARIANTS["B_plus_ip_pattern_false_positives"] = core + ip_fp
if ports_dst: VARIANTS["C1_plus_dst_port"] = core + ports_dst
if ports_src: VARIANTS["C2_plus_src_port"] = core + ports_src
if ports: VARIANTS["C3_plus_both_ports"] = core + ports
if ports or ip_fp: VARIANTS["D_old_policy_all"] = core + ip_fp + ports
if ports: VARIANTS["E_ports_only"] = list(ports)
_fv = os.environ.get("FP_VARIANTS")
if _fv:
    _keep = {t.strip().upper() for t in _fv.split(",") if t.strip()} | {"A"}
    VARIANTS = {k: v for k, v in VARIANTS.items() if k.split("_")[0].upper() in _keep}
MODEL_KINDS = [m.strip() for m in os.environ.get("FP_MODELS", "default_DT,saved_params_DT").split(",") if m.strip()]
print("\nvariants:", {k: len(v) for k, v in VARIANTS.items()}, "| models:", MODEL_KINDS)

## 2 · Re-create the exact split and read only the needed rows

In [ ]:
t0 = time.perf_counter()
le, y_all = fp.read_target(DEFAULT_DATASET_PATH, TARGET_COL)
class_names = list(le.classes_)
part = fp.make_partition(y_all, FP_POOL, FP_EVAL_LIMIT, SEED)
all_cols = list(dict.fromkeys(sum(VARIANTS.values(), [])))
rows_df = fp.read_rows(DEFAULT_DATASET_PATH, all_cols, part)
Xpool, Xv, Xt = rows_df["pool"], rows_df["val"], rows_df["test"]
ypool, yv, yt = y_all[Xpool.index.values], y_all[Xv.index.values], y_all[Xt.index.values]
rare_classes, extremely_rare_classes = infer_rare_classes(ypool)
print(f"{len(y_all):,} rows in file | train pool {len(Xpool):,} | validation {len(Xv):,} | test {len(Xt):,} | {len(all_cols)} columns | loaded in {time.perf_counter()-t0:.0f}s")
print("classes:", class_names)

# sanity: the test rows must be a subset of the main pipeline's test split (same partition)
_tr = set(part["train_all"].tolist()); assert not (_tr & set(Xt.index.tolist())), "test rows leaked into train!"
print("no overlap between train and test rows")

## 3 · Train and score  *(resumable; partial results saved after every run)*

In [ ]:
OUT = Path("feature_policy_ablation") / (ACTIVE_DATASET + FP_TAG); OUT.mkdir(parents=True, exist_ok=True)
cfg = {"variants": {k: len(v) for k, v in VARIANTS.items()}, "models": MODEL_KINDS, "tag": FP_TAG, "k_base_n": K_BASE_N, "train_rows": FP_TRAIN_LIMIT, "pool": FP_POOL, "eval": FP_EVAL_LIMIT,
       "best_k": BEST_K, "params": json.loads(json.dumps(BEST_PARAMS, default=str)), "seed_base": SEED + 900}
cfg_file, raw_file, prog_file = OUT / "run_config.json", OUT / "feature_policy_raw.csv", OUT / "progress.txt"
old = json.loads(cfg_file.read_text()) if cfg_file.exists() else None
if raw_file.exists() and old != cfg:
    arch = OUT / ("previous_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")); arch.mkdir()
    for f in OUT.glob("*"):
        if f.is_file() and f.name != "column_groups.json": shutil.move(str(f), arch / f.name)
    print("settings changed -> old results archived to", arch.name)
cfg_file.write_text(json.dumps(cfg, indent=2, default=str))

rows = pd.read_csv(raw_file).to_dict("records") if raw_file.exists() else []
pc_rows = pd.read_csv(OUT / "feature_policy_per_class.csv").to_dict("records") if (OUT / "feature_policy_per_class.csv").exists() else []
done = {(int(r["repeat"]), r["variant"], r["model"]) for r in rows}
PORTS_SET, IPFP_SET = set(ports), set(ip_fp)

def exported_kb(nodes, k): return (11 * nodes + 13 * k) / 1024
def log(msg):
    print(msg, flush=True)
    with open(prog_file, "a", encoding="utf-8") as f: f.write(datetime.datetime.now().strftime("%H:%M:%S ") + msg + "\n")

total = FP_REPEATS * len(VARIANTS) * len(MODEL_KINDS); t_start = time.perf_counter(); n_new = 0
for rep in range(FP_REPEATS):
    seed = SEED + 900 + rep
    Xtr_all, ytr = stratified_sample(Xpool, ypool, FP_TRAIN_LIMIT, seed)
    for vname, cols in VARIANTS.items():
        for mkind in MODEL_KINDS:
            if (rep, vname, mkind) in done: continue
            t0 = time.perf_counter()
            if mkind == "default_DT": pipe = build_pipeline({}, "all", seed)
            else: pipe = build_pipeline(BEST_PARAMS, len(cols) if vname == 'E_ports_only' else max(1, min(len(cols), round(BEST_K * len(cols) / K_BASE_N))), seed)   # same FRACTION of features as the saved model (all features for ports-only)
            pipe.fit(Xtr_all[cols], ytr)
            pv, pt = pipe.predict(Xv[cols]), pipe.predict(Xt[cols])
            mv = compute_enhanced_metrics(yv, pv, rare_classes, extremely_rare_classes); mt = compute_enhanced_metrics(yt, pt, rare_classes, extremely_rare_classes)
            clf = pipe.named_steps["clf"]; nodes = int(clf.tree_.node_count)
            sel = pipe.named_steps["selector"].get_support(indices=True); used = [cols[i] for i in sel]
            imp = clf.feature_importances_
            share = lambda S: float(sum(imp[j] for j, c in enumerate(used) if c in S))
            rows.append({"variant": vname, "model": mkind, "repeat": rep, "n_features": len(cols), "n_used": len(used), "train_rows": len(Xtr_all),
                         "val_mcc": mv["mcc"], "val_macro_f1": mv["macro_f1"], "test_mcc": mt["mcc"], "test_macro_f1": mt["macro_f1"],
                         "test_accuracy": mt["accuracy"], "test_minority_recall": mt["minority_recall"], "test_small_class_f1": mt["small_class_f1"],
                         "nodes": nodes, "exported_kb": exported_kb(nodes, len(used)),
                         "ports_kept": sum(c in PORTS_SET for c in used), "ip_fp_kept": sum(c in IPFP_SET for c in used),
                         "port_importance_share": share(PORTS_SET), "ip_fp_importance_share": share(IPFP_SET), "seconds": time.perf_counter() - t0})
            _, rc, f1c, sup = precision_recall_fscore_support(yt, pt, labels=np.arange(len(class_names)), zero_division=0)
            for i, c in enumerate(class_names):
                pc_rows.append({"variant": vname, "model": mkind, "repeat": rep, "class_name": c, "recall": rc[i], "f1": f1c[i], "support": int(sup[i])})
            pd.DataFrame(rows).to_csv(raw_file, index=False); pd.DataFrame(pc_rows).to_csv(OUT / "feature_policy_per_class.csv", index=False)
            n_new += 1; left = (time.perf_counter() - t_start) / n_new * (total - len(rows))
            log(f"rep {rep+1}/{FP_REPEATS} | {vname:36s} {mkind:16s} test MCC {mt['mcc']:.4f} macro-F1 {mt['macro_f1']:.4f} | nodes {nodes:>7,} | ports share {share(PORTS_SET):.2f} | ~{left/60:4.0f} min left")
print("\nfinished:", len(rows), "runs ->", OUT)

## 4 · Results

In [ ]:
raw = pd.read_csv(raw_file)
agg = raw.groupby(["model", "variant"]).agg(n=("repeat", "size"), n_features=("n_features", "first"),
        test_mcc=("test_mcc", "mean"), test_mcc_std=("test_mcc", "std"), test_macro_f1=("test_macro_f1", "mean"), test_macro_f1_std=("test_macro_f1", "std"),
        test_accuracy=("test_accuracy", "mean"), minority_recall=("test_minority_recall", "mean"), small_class_f1=("test_small_class_f1", "mean"),
        nodes=("nodes", "median"), exported_kb=("exported_kb", "median"), ports_kept=("ports_kept", "mean"), port_importance_share=("port_importance_share", "mean"),
        ip_fp_importance_share=("ip_fp_importance_share", "mean")).reset_index()
ref = agg[agg.variant == "A_current_policy"].set_index("model")
for c in ["test_mcc", "test_macro_f1", "minority_recall", "nodes"]:
    agg[f"d_{c}_vs_A"] = agg.apply(lambda r: r[c] - ref.loc[r["model"], c], axis=1)
agg.to_csv(OUT / "feature_policy_summary.csv", index=False)
display(agg[agg.model == "saved_params_DT"].drop(columns=["model"]).round(4))
print("\ndefault decision tree:")
display(agg[agg.model == "default_DT"].drop(columns=["model"]).round(4))

# paired differences vs A (same repeat = same training rows)
rng = np.random.default_rng(0); pr = []
for mk in raw.model.unique():
    a = raw[(raw.model == mk) & (raw.variant == "A_current_policy")].set_index("repeat")
    for v in raw.variant.unique():
        if v == "A_current_policy": continue
        b = raw[(raw.model == mk) & (raw.variant == v)].set_index("repeat")
        idx = a.index.intersection(b.index)
        for m in ["test_mcc", "test_macro_f1", "test_minority_recall", "nodes"]:
            d = (b.loc[idx, m] - a.loc[idx, m]).values
            if len(d) == 0: continue
            bs = rng.choice(d, size=(5000, len(d)), replace=True).mean(axis=1)
            pr.append({"model": mk, "variant": v, "metric": m, "n_pairs": len(d), "mean_diff_vs_A": d.mean(), "ci95_low": np.percentile(bs, 2.5), "ci95_high": np.percentile(bs, 97.5)})
paired = pd.DataFrame(pr); paired.to_csv(OUT / "feature_policy_paired.csv", index=False)

In [ ]:
# ---- figure 1: MCC / macro-F1 / port importance by feature policy ----------------------------
order = list(VARIANTS)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3), constrained_layout=True)
for ax, (col, sdcol, title) in zip(axes[:2], [("test_mcc", "test_mcc_std", "test MCC"), ("test_macro_f1", "test_macro_f1_std", "test macro-F1")]):
    w = .38
    for j, (mk, lab, colr) in enumerate([("saved_params_DT", "saved hyper-parameters", "#1f77b4"), ("default_DT", "default tree", "#999999")]):
        d = agg[agg.model == mk].set_index("variant").reindex(order)
        ax.bar(np.arange(len(order)) + (j - .5) * w, d[col], w, yerr=d[sdcol].fillna(0), capsize=2, label=lab, color=colr)
    ax.set_xticks(range(len(order)), [o.split("_")[0] for o in order]); ax.set_ylabel(title); ax.set_title(title + " by feature set")
    lo = agg[col].min(); ax.set_ylim(max(0, lo - .05), min(1.0, agg[col].max() + .03)); ax.legend(fontsize=7)
for j, (mk, lab, colr) in enumerate([("default_DT", "default tree", "#d95f02"), ("saved_params_DT", "saved-parameter tree (after SelectKBest)", "#f4b183")]):
    d = agg[agg.model == mk].set_index("variant").reindex(order)
    axes[2].bar(np.arange(len(order)) + (j - .5) * .38, d["port_importance_share"], .38, label=lab, color=colr)
axes[2].set_xticks(range(len(order)), [o.split("_")[0] for o in order]); axes[2].set_ylim(0, 1); axes[2].set_ylabel("share of the tree's feature importance on PORTS"); axes[2].legend(fontsize=7)
axes[2].set_title("how much the tree relies on the ports")
fig.suptitle(f"{ACTIVE_DATASET}: feature-policy ablation  (A=leakage-safe features, B=+ip-pattern features, C=+ports, D=A+B+ports, E=ports only)", fontsize=10)
fig.savefig(FIGURES_DIR / f"feature_policy_ablation{FP_TAG}.png", dpi=300, bbox_inches="tight"); plt.show()

# ---- figure 2: per-class recall, saved-params tree ------------------------------------------
pc = pd.read_csv(OUT / "feature_policy_per_class.csv"); pc = pc[pc.model == "saved_params_DT"]
piv = pc.groupby(["class_name", "variant"])["recall"].mean().unstack("variant").reindex(index=class_names, columns=order)
fig, ax = plt.subplots(figsize=(1.0 * len(order) + 3, .5 * len(class_names) + 1.8))
im = ax.imshow(piv.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto"); ax.grid(False)
ax.set_xticks(range(len(order)), [o.split("_")[0] for o in order]); ax.set_yticks(range(len(class_names)), class_names)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]): ax.text(j, i, f"{piv.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("per-class test recall by feature set (saved-parameter tree)"); fig.colorbar(im, ax=ax, shrink=.8)
fig.savefig(FIGURES_DIR / f"feature_policy_per_class_recall{FP_TAG}.png", dpi=300, bbox_inches="tight"); plt.show()
piv.to_csv(OUT / "feature_policy_per_class_recall_table.csv")

## 5 · Contribution of each feature group

In [ ]:
def get(mk, v, col):
    r = agg[(agg.model == mk) & (agg.variant == v)]
    return float(r[col].iloc[0]) if len(r) else np.nan

lines = []
for mk, label in [("saved_params_DT", "tree with the saved hyper-parameters"), ("default_DT", "default tree")]:
    A_m, A_f = get(mk, "A_current_policy", "test_mcc"), get(mk, "A_current_policy", "test_macro_f1")
    lines.append(f"### {label}\n")
    lines.append(f"reference A (leakage-safe features): test MCC {A_m:.4f}, macro-F1 {A_f:.4f}\n")
    tbl = []
    for v, text in [("B_plus_ip_pattern_false_positives", "+ the 3 features removed by the ip pattern"), ("C1_plus_dst_port", "+ destination port"),
                    ("C2_plus_src_port", "+ source port"), ("C3_plus_both_ports", "+ both ports"), ("D_old_policy_all", "A + B + both ports")]:
        if np.isnan(get(mk, v, "test_mcc")): continue
        tbl.append({"feature set": text, "ΔMCC vs A": get(mk, v, "test_mcc") - A_m, "Δmacro-F1 vs A": get(mk, v, "test_macro_f1") - A_f,
                    "Δnodes": get(mk, v, "nodes") - get(mk, "A_current_policy", "nodes")})
    display(pd.DataFrame(tbl).round(4))
E_f1, E_m = get("default_DT", "E_ports_only", "test_macro_f1"), get("default_DT", "E_ports_only", "test_mcc")
share = get("default_DT", "C3_plus_both_ports", "port_importance_share")
print(f"ports ALONE (default tree): test MCC {E_m:.3f}, macro-F1 {E_f1:.3f}   |   share of the tree's importance on ports when they are allowed: {share:.2f}")
_pk = get("saved_params_DT", "C3_plus_both_ports", "ports_kept")
if _pk == 0 and share > 0.2:
    print("NOTE: SelectKBest (linear F-test) removed the ports in the saved-parameter tree, but the default tree uses them heavily -> the shortcut is real; the saved-parameter rows understate it.")
_dn = get("default_DT", "C3_plus_both_ports", "nodes") - get("default_DT", "A_current_policy", "nodes")
print(f"tree size change when ports are added (default tree): {_dn:+.0f} nodes   (a much SMALLER tree with a much HIGHER score is a typical shortcut signature)")
gd_ports = get("default_DT", "C3_plus_both_ports", "test_macro_f1") - get("default_DT", "A_current_policy", "test_macro_f1")
gd_ipfp = get("default_DT", "B_plus_ip_pattern_false_positives", "test_macro_f1") - get("default_DT", "A_current_policy", "test_macro_f1")
print("\nReading guide (indicative):")
print(" * ports ALONE score high AND the tree puts a large share of its importance on them -> strong shortcut signal.")
print(" * ports ALONE score high but add nothing on top of the other features (low share, no gain) -> redundant, not a decisive shortcut; keeping them out is still the conservative policy.")
print(" * a gain from the 3 'ip'-pattern features means the name pattern removes useful information.")
print(" * if D is close to C3, any gain of D comes from the ports, not from the extra features.")
print(f"macro-F1 gain (default tree) from the 3 'ip'-pattern features: {gd_ipfp:+.4f}   |   from the ports: {gd_ports:+.4f}")